In [1]:
import numpy as np

# Carregar dados do arquivo 'problemdata.npz'
problemdata = np.load('problemdata.npz')

# Lista de variáveis a serem extraídas
variaveis = list(problemdata.keys())

# Atribuir os valores correspondentes às variáveis
for variavel in variaveis:
    globals()[variavel] = problemdata[variavel]
    
# Lista de variáveis a serem convertidas para inteiros
variaveis = [I, J, K, E, S, N1, N2, N3, Q, M]

# Converter todas as variáveis para inteiros de uma vez
I, J, K, E, S, N1, N2, N3, Q, M = tuple(map(int, variaveis))

In [7]:
import gurobipy as grb
import csv

# Criação do modelo
modelo = grb.Model(
    """Otimização de rede de cadeia de abastecimento de pistache com 
    realimentação"""
)

# Variáveis de decisão positivas: fluxos de produtos
X = modelo.addVars(int(I), int(J), vtype=grb.GRB.CONTINUOUS, name="X", lb=0.)
Go = modelo.addVars(J, K, vtype=grb.GRB.CONTINUOUS, name="Go", lb=0.)
Gr = modelo.addVars(J, E, vtype=grb.GRB.CONTINUOUS, name="Gr", lb=0.)
Gw = modelo.addVars(J, Q, vtype=grb.GRB.CONTINUOUS, name="Gw", lb=0.)
O = modelo.addVars(E, N2, vtype=grb.GRB.CONTINUOUS, name="O", lb=0.)
Oc = modelo.addVars(E, S, vtype=grb.GRB.CONTINUOUS, name="Oc", lb=0.)
Ow = modelo.addVars(E, Q, vtype=grb.GRB.CONTINUOUS, name="Ow", lb=0.)
L = modelo.addVars(S, N3, vtype=grb.GRB.CONTINUOUS, name="L", lb=0.)
P = modelo.addVars(K, N1, vtype=grb.GRB.CONTINUOUS, name="P", lb=0.)
D = modelo.addVars(Q, M, vtype=grb.GRB.CONTINUOUS, name="D", lb=0.)

# Variáveis binárias: indicadores de ativação
U = modelo.addVars(J, vtype=grb.GRB.BINARY, name="U")
Y = modelo.addVars(Q, vtype=grb.GRB.BINARY, name="Y")
W = modelo.addVars(K, vtype=grb.GRB.BINARY, name="W")
R = modelo.addVars(E, vtype=grb.GRB.BINARY, name="R")
V = modelo.addVars(S, vtype=grb.GRB.BINARY, name="V")

# Custo de abertura de instalações
z1 = (grb.quicksum(Fu[j]*U[j] for j in range(J))
      + grb.quicksum(Fy[q]*Y[q] for q in range(Q))
      + grb.quicksum(Fw[k]*W[k] for k in range(K))
      + grb.quicksum(Fr[e]*R[e] for e in range(E))
      + grb.quicksum(Fv[s]*V[s] for s in range(S)))

# Custo de produção
z2 = (grb.quicksum(CI[i]*X[i,j] for i in range(I) for j in range(J))
      + grb.quicksum(Cu1[j]*Go[j,k] for j in range(J) for k in range(K))
      + grb.quicksum(Cu2[j]*Gr[j,e] for j in range(J) for e in range(E))
      + grb.quicksum(Cy[q]*D[q,m] for q in range(Q) for m in range(M))
      + grb.quicksum(Cw[k]*P[k,n1] for k in range(K) for n1 in range(N1))
      + grb.quicksum(Cr[e]*O[e,n2] for e in range(E) for n2 in range(N2))
      + grb.quicksum(Cr[e]*Oc[e,s] for e in range(E) for s in range(S))
      + grb.quicksum(Cv[s]*L[s,n3] for s in range(S) for n3 in range(N3)))

# Custos de transporte
z3 = (grb.quicksum(CX[i,j]*X[i,j] for i in range(I) for j in range(J))
      + grb.quicksum(CK[j,k]*Go[j,k] for j in range(J) for k in range(K))
      + grb.quicksum(CE[j,e]*Gr[j,e] for j in range(J) for e in range(E))
      + grb.quicksum(CJ[j,q]*Gw[j,q] for j in range(J) for q in range(Q))
      + grb.quicksum(CS[e,s]*Oc[e,s] for e in range(E) for s in range(S))
      + grb.quicksum(CN[e,n2]*O[e,n2] for e in range(E) for n2 in range(N2))
      + grb.quicksum(CQ[e,q]*Ow[e,q] for e in range(E) for q in range(Q))
      + grb.quicksum(Cl[s, n3]*L[s,n3] for s in range(S) for n3 in range(N3))
      + grb.quicksum(Cp[k,n1]*P[k,n1] for k in range(K) for n1 in range(N1))
      + grb.quicksum(Cd[q,m]*D[q,m] for q in range(Q) for m in range(M)))

    
# Definindo a função objetivo
modelo.setObjective(z1 + z2 + z3, grb.GRB.MINIMIZE)

# Restrição de capacidade
modelo.addConstrs(
    (grb.quicksum(X[i,j] for j in range(J)) <= Cpa[i] for i in range(I)), 
    name="Eq.(4)"
)
modelo.addConstrs(
    (grb.quicksum(X[i,j] for i in range(I)) <= Cpu[j]*U[j] for j in range(J)), 
    name="Eq.(5)"
)
modelo.addConstrs(
    (grb.quicksum(Ow[e,q] for e in range(E))
     + grb.quicksum(Gw[j,q] for j in range(J)) 
     <= Cpy[q]*Y[q] for q in range(Q)), name="Eq.(6)"
)
modelo.addConstrs(
    (grb.quicksum(Go[j,k] for j in range(J)) <= Cpw[k]*W[k] for k in range(K)),
    name="Eq.(7)"
)
modelo.addConstrs(
    (grb.quicksum(Gr[j,e] for j in range(J)) <= Cpr[e]*R[e] for e in range(E)), 
    name="Eq.(8)"
)
modelo.addConstrs(
    (grb.quicksum(Oc[e,s] for e in range(E)) <= Cpv[s]*V[s] for s in range(S)),
    name="Eq.(9)"
)
modelo.addConstrs(
    (grb.quicksum(Go[j,k] for k in range(K))
     + grb.quicksum(Gr[j,e] for e in range(E))
     + grb.quicksum(Gw[j,q] for q in range(Q))
     <= grb.quicksum(X[i,j] for i in range(I)) for j in range(J)),
    name="Eq.(10)"
)
modelo.addConstrs(
    (grb.quicksum(Go[j,k] for k in range(K))
     == (1-beta)*theta1*grb.quicksum(X[i,j] for i in range(I)) for j in range(J)),
    name="Eq.(11)"
)
modelo.addConstrs(
    (grb.quicksum(Gr[j,e] for e in range(E))
     == (1-beta)*theta2*grb.quicksum(X[i,j] for i in range(I)) for j in range(J)),
    name="Eq.(12)"
)
modelo.addConstrs(
    (grb.quicksum(Gw[j,q] for q in range(Q))
     == theta3*grb.quicksum(X[i,j] for i in range(I)) for j in range(J)), 
    name="Eq.(13)"
)
modelo.addConstrs(
    (grb.quicksum(P[k,n1] for n1 in range(N1))
     <= gammak*grb.quicksum(Go[j,k] for j in range(J)) for k in range(K)),
    name="Eq.(14)"
)
modelo.addConstrs(
    (grb.quicksum(O[e,n2] for n2 in range(N2)) 
     + grb.quicksum(Oc[e,s] for s in range(S)) 
     <= (1-lamb)*grb.quicksum(Gr[j,e] for j in range(J)) for e in range(E)), 
    name="Eq.(15)"
)
modelo.addConstrs(
    (grb.quicksum(Ow[e,q] for q in range(Q))
     <= lamb*grb.quicksum(Gr[j,e] for j in range(J)) for e in range(E)), 
    name="Eq.(16)"
)
modelo.addConstrs(
    (grb.quicksum(L[s,n3] for n3 in range(N3)) 
     <= gammas*grb.quicksum(Oc[e,s] for e in range(E)) for s in range(S)),
    name="Eq.(17)"
)
modelo.addConstrs(
    (grb.quicksum(D[q,m] for m in range(M))
     <= gammaq*(grb.quicksum(Gw[j,q] for j in range(J)) 
                + grb.quicksum(Ow[e,q] for e in range(E))) for q in range(Q)), 
    name="Eq.(18)"
)
modelo.addConstrs(
    (grb.quicksum(P[k,n1] for k in range(K)) >= Dp[n1] for n1 in range(N1)), 
    name="Eq.(19)"
)
modelo.addConstrs(
    (grb.quicksum(O[e,n2] for e in range(E)) >= Du[n2] for n2 in range(N2)), 
    name="Eq.(20)"
)
modelo.addConstrs(
    (grb.quicksum(L[s,n3] for s in range(S)) >= Ds[n3] for n3 in range(N3)), 
    name="Eq.(21)"
)
modelo.addConstrs(
    (grb.quicksum(D[q,m] for q in range(Q)) >= Dc[m] for m in range(M)),
    name="Eq.(22)"
)

# Resolvendo o modelo
modelo.optimize()
variaveis_decisao = modelo.getVars()

with open('exato.csv', 'w', newline='') as arquivo_csv:
    escritor_csv = csv.writer(arquivo_csv)
    escritor_csv.writerow(['Variável', 'Valor'])
        
    for var in variaveis_decisao:
        escritor_csv.writerow([var.VarName, var.x])

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU E3-1240 V2 @ 3.40GHz, instruction set [SSE2|AVX]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Academic license 2472240 - for non-commercial use only - registered to wm___@gmail.com
Optimize a model with 29 rows, 27 columns and 88 nonzeros
Model fingerprint: 0xa64ab476
Variable types: 21 continuous, 6 integer (6 binary)
Coefficient statistics:
  Matrix range     [7e-02, 4e+04]
  Objective range  [2e+01, 6e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [4e+01, 1e+05]
Presolve removed 26 rows and 22 columns
Presolve time: 0.00s
Presolved: 3 rows, 5 columns, 7 nonzeros
Variable types: 3 continuous, 2 integer (2 binary)
Found heuristic solution: objective 1610991.1256



Root relaxation: cutoff, 0 iterations, 0.00 seconds (0.00 work units)

Explored 1 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 8 (of 8 available processors)

Solution count 1: 1.61099e+06 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.610991125597e+06, best bound 1.610991125597e+06, gap 0.0000%
